# Intelligent Road Scene Understanding & Multi-Class Object Recognition using CNNs
**College PBL project: CPU-friendly (no GPU needed)**

### Problem
A self-driving or driver-assistance system must (1) *recognise traffic signs* and (2) *understand what is on the road* (cars, people, traffic lights...).

### What we build
| Part | Task | Method | Data |
|---|---|---|---|
| 1 | Traffic-sign recognition (43 classes) | Our own CNN trained from scratch | GTSRB (German Traffic Sign Benchmark) |
| 2 | Road-scene object detection | Pre-trained CNN detector (Faster R-CNN MobileNetV3) | COCO weights, any road photo |
| 3 | Scene summary | Combine detections into a short safety description | - |

**Runtime on a normal laptop CPU:** about 10-15 min (mostly Part 1 training).

In [ ]:
# Run once if packages are missing (then restart kernel):
# %pip install torch torchvision matplotlib scikit-learn pillow

## 1. Setup

In [ ]:
import random, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ---- Settings you can change ----
IMG = 32
EPOCHS = 8
TRAIN_LIMIT = None
BATCH = 128

## 2. Dataset: GTSRB
43 classes of German traffic signs (~26,640 training and 12,630 test images). It downloads automatically (about 300 MB) the first time.

In [ ]:
CLASS_NAMES = ["Speed limit 20","Speed limit 30","Speed limit 50","Speed limit 60","Speed limit 70",
 "Speed limit 80","End speed limit 80","Speed limit 100","Speed limit 120","No passing",
 "No passing (>3.5t)","Right-of-way at intersection","Priority road","Yield","Stop",
 "No vehicles","No vehicles >3.5t","No entry","General caution","Dangerous curve left",
 "Dangerous curve right","Double curve","Bumpy road","Slippery road","Road narrows right",
 "Road work","Traffic signals","Pedestrians","Children crossing","Bicycles crossing",
 "Ice/snow","Wild animals","End all limits","Turn right ahead","Turn left ahead",
 "Ahead only","Straight or right","Straight or left","Keep right","Keep left",
 "Roundabout","End no passing","End no passing (>3.5t)"]

tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor()])
train_raw = datasets.GTSRB("data", split="train", download=True, transform=tf)
test_raw  = datasets.GTSRB("data", split="test",  download=True, transform=tf)
print("train images:", len(train_raw), "| test images:", len(test_raw))

In [ ]:
def to_tensors(ds, limit=None):
    """Decode all images once into memory so every epoch is fast on CPU."""
    idx = torch.randperm(len(ds))[:limit].tolist() if limit else range(len(ds))
    X, y = [], []
    for i in idx:
        img, label = ds[i]
        X.append(img); y.append(label)
    X = (torch.stack(X) - 0.5) / 0.5
    return X, torch.tensor(y)

t0 = time.time()
X_all, y_all = to_tensors(train_raw, TRAIN_LIMIT)
X_test, y_test = to_tensors(test_raw)
print(f"Loaded in {time.time()-t0:.0f}s  ->  {X_all.shape}, {X_test.shape}")

full = TensorDataset(X_all, y_all)
n_val = int(0.1 * len(full))
train_ds, val_ds = random_split(full, [len(full) - n_val, n_val], generator=torch.Generator().manual_seed(SEED))
train_dl = DataLoader(train_ds, BATCH, shuffle=True)
val_dl   = DataLoader(val_ds, BATCH)
test_dl  = DataLoader(TensorDataset(X_test, y_test), BATCH)

### Explore the data

In [ ]:
def show(x):
    return (x * 0.5 + 0.5).permute(1, 2, 0).clamp(0, 1).numpy()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, i in zip(axes.flat, np.random.choice(len(X_all), 16, replace=False)):
    ax.imshow(show(X_all[i])); ax.set_title(CLASS_NAMES[y_all[i]][:16], fontsize=7); ax.axis("off")
plt.suptitle("Sample training images"); plt.show()

counts = np.bincount(y_all.numpy(), minlength=43)
plt.figure(figsize=(12, 3)); plt.bar(range(43), counts)
plt.xlabel("class id"); plt.ylabel("images"); plt.title("Class distribution (imbalanced!)"); plt.show()

## 3. The CNN model
Three convolution blocks (**Conv -> BatchNorm -> ReLU -> MaxPool**) extract features such as edges, then shapes, then sign patterns; two fully-connected layers with **Dropout** classify into 43 classes.

In [ ]:
class SignCNN(nn.Module):
    def __init__(self, n_classes=43):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

model = SignCNN().to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

## 4. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(dl, train):
    model.train(train)
    loss_sum = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return loss_sum / n, correct / n

hist = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}
for ep in range(EPOCHS):
    t0 = time.time()
    trl, tra = run_epoch(train_dl, True)
    val, vaa = run_epoch(val_dl, False)
    for k, v in zip(hist, (trl, val, tra, vaa)): hist[k].append(v)
    print(f"Epoch {ep+1}/{EPOCHS}  train loss {trl:.3f} acc {tra:.3f} | val loss {val:.3f} acc {vaa:.3f}  ({time.time()-t0:.0f}s)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist["tr_acc"], label="train"); ax[1].plot(hist["va_acc"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

## 5. Evaluation on the unseen test set

In [ ]:
model.eval()
preds, trues = [], []
with torch.no_grad():
    for x, y in test_dl:
        preds.append(model(x.to(device)).argmax(1).cpu()); trues.append(y)
preds, trues = torch.cat(preds).numpy(), torch.cat(trues).numpy()

test_acc = (preds == trues).mean()
print(f"TEST ACCURACY: {test_acc*100:.2f}%")
rep = classification_report(trues, preds, labels=range(43), zero_division=0, output_dict=True)
print(f"Macro precision {rep['macro avg']['precision']:.3f} | recall {rep['macro avg']['recall']:.3f} | F1 {rep['macro avg']['f1-score']:.3f}")

f1 = {i: rep[str(i)]["f1-score"] for i in range(43) if str(i) in rep}
print("\nWeakest 5 classes (by F1):")
for i, s in sorted(f1.items(), key=lambda kv: kv[1])[:5]:
    print(f"  {CLASS_NAMES[i]:32s} F1={s:.2f}")

In [ ]:
cm = confusion_matrix(trues, preds, labels=range(43))
plt.figure(figsize=(9, 8)); plt.imshow(cm, cmap="Blues")
plt.colorbar(); plt.xlabel("Predicted class"); plt.ylabel("True class")
plt.title("Confusion matrix (43 traffic-sign classes)"); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
print("Most confused pairs (true -> predicted):")
for flat in np.argsort(off, axis=None)[::-1][:5]:
    a, b = divmod(flat, 43)
    if off[a, b] > 0:
        print(f"  {CLASS_NAMES[a]}  ->  {CLASS_NAMES[b]}   ({off[a, b]} times)")

In [ ]:
wrong = np.where(preds != trues)[0]
print(f"{len(wrong)} wrong out of {len(trues)}")
if len(wrong):
    fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
    for ax, i in zip(axes.flat, np.random.choice(wrong, min(10, len(wrong)), replace=False)):
        ax.imshow(show(X_test[i]))
        ax.set_title(f"true: {CLASS_NAMES[trues[i]][:18]}\npred: {CLASS_NAMES[preds[i]][:18]}", fontsize=8)
        ax.axis("off")
    plt.suptitle("Misclassified test images"); plt.show()

### Classify your own traffic-sign photo
Put a cropped sign image in the notebook folder and set `SIGN_PATH`.

In [ ]:
import os
from PIL import Image
SIGN_PATH = "my_sign.jpg"

def predict_sign(path, topk=3):
    img = Image.open(path).convert("RGB")
    x = (tf(img) - 0.5) / 0.5
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(x[None].to(device)), 1)[0].cpu()
    top = p.topk(topk)
    plt.imshow(img); plt.axis("off"); plt.show()
    for s, i in zip(top.values, top.indices):
        print(f"{CLASS_NAMES[i]:32s} {s*100:5.1f}%")

if os.path.exists(SIGN_PATH):
    predict_sign(SIGN_PATH)
else:
    print(f"Add an image called '{SIGN_PATH}' to try this cell.")

In [ ]:
torch.save(model.state_dict(), "sign_cnn.pt")
print("Saved model to sign_cnn.pt")

## 6. Road-scene object detection (pre-trained CNN)
Training a detector from scratch needs big data and a GPU, so we use **transfer learning**: a Faster R-CNN (MobileNetV3 backbone) already trained on COCO. It recognises cars, buses, trucks, people, bicycles, motorcycles, traffic lights and stop signs, with no training needed.

Put any road/street photo in the notebook folder as `road.jpg` (or change `IMAGE_PATH`). The first run downloads the weights (about 75 MB).

In [ ]:
from torchvision.models.detection import (fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights)
import torchvision.transforms.functional as TF
import matplotlib.patches as patches

weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
det_model = fasterrcnn_mobilenet_v3_large_fpn(weights=weights).eval()
COCO = weights.meta["categories"]

ROAD_OBJECTS = {"person", "bicycle", "car", "motorcycle", "bus", "truck", "traffic light", "stop sign"}
VULNERABLE = {"person", "bicycle", "motorcycle"}

In [ ]:
IMAGE_PATH = "road.jpg"
THRESH = 0.5

def analyze_road_image(path, thr=THRESH):
    img = Image.open(path).convert("RGB")
    with torch.no_grad():
        out = det_model([TF.to_tensor(img)])[0]
    found = []
    for box, lab, sc in zip(out["boxes"], out["labels"], out["scores"]):
        name = COCO[int(lab)]
        if sc >= thr and name in ROAD_OBJECTS:
            found.append((name, float(sc), box.tolist()))
    fig, ax = plt.subplots(figsize=(11, 6)); ax.imshow(img); ax.axis("off")
    for name, sc, (x1, y1, x2, y2) in found:
        color = "red" if name in VULNERABLE else "yellow"
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2))
        ax.text(x1, max(y1-4, 0), f"{name} {sc:.2f}", color="black", fontsize=8, bbox=dict(facecolor=color, alpha=0.8, pad=1))
    plt.title("Detected road objects"); plt.show()
    return found

if os.path.exists(IMAGE_PATH):
    detections = analyze_road_image(IMAGE_PATH)
else:
    detections = []
    print(f"Add a road photo called '{IMAGE_PATH}' next to this notebook and re-run.")

## 7. Scene understanding summary
Turn raw detections into a short, human-readable description with a simple safety rule.

In [ ]:
def scene_summary(found):
    counts = {}
    for name, _, _ in found:
        counts[name] = counts.get(name, 0) + 1
    if not counts:
        return "No road objects detected."
    text = "Scene contains: " + ", ".join(f"{n} x {k}" for k, n in counts.items()) + "."
    vru = sum(counts.get(k, 0) for k in VULNERABLE)
    if vru:
        text += f" CAUTION: {vru} vulnerable road user(s) nearby, so slow down."
    if "traffic light" in counts:
        text += " Traffic light ahead."
    if "stop sign" in counts:
        text += " Stop sign ahead."
    if counts.get("car", 0) + counts.get("truck", 0) + counts.get("bus", 0) >= 5:
        text += " Heavy traffic."
    return text

print(scene_summary(detections))

## 8. Results & discussion
*(Fill in with your own numbers after running.)*

| Item | Result |
|---|---|
| Test accuracy (43 classes) | ___ % |
| Macro F1 | ___ |
| Training time on CPU | ___ min |
| Most confused signs | ___ |

**Observations to discuss**
- Similar-looking signs (e.g. different speed limits) are confused the most. Digits are tiny at 32x32.
- The dataset is imbalanced, so rare classes score lower.
- Train vs validation curves show whether the model overfits.

**Limitations:** the detector cannot read *which* traffic sign it sees (COCO only has "stop sign"). The two parts are separate models; images are only German signs in good-ish lighting.

**Future work:** (1) run a sign detector (e.g. YOLO trained on GTSRB-detection) and feed crops into our CNN, (2) add semantic segmentation, (3) track objects over video, (4) improve robustness with augmentation.

### Reproducible outputs
Run **Kernel → Restart & Run All** (or **Runtime → Run all** in Google Colab) to generate the training metrics, plots, confusion matrix, misclassification examples, saved model, and road-scene detection output. The notebook intentionally does not contain fabricated metrics; outputs depend on the downloaded GTSRB dataset and model weights.